Fit T Regression using test7_3.csv and testout_7.3.csv

Method 1

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import t
from scipy.optimize import minimize
from scipy.special import gammaln

In [7]:
data = pd.read_csv('../testfiles/data/test7_3.csv')
x = data[["x1", "x2", "x3"]].to_numpy()
y = data["y"].to_numpy()
n = len(y)

In [8]:
# OLS for intercept and betas
X_ols = np.column_stack([np.ones(n), x])
beta_ols = np.linalg.lstsq(X_ols, y, rcond=None)[0]
residuals = y - X_ols.dot(beta_ols)
sigma_init = np.std(residuals)
nu_init = 5.0  # degrees of freedom, will optimize later

In [5]:
# Initial parameter vector: [Alpha, B1, B2, B3, log(sigma), log(nu)]
initial_params = np.concatenate([beta_ols, [np.log(sigma_init), np.log(nu_init)]])

In [21]:
# negative log likelihood
def neg_log_likelihood(params):
    alpha, b1, b2, b3, sigma, nu = params
    sigma = np.exp(sigma)
    nu = np.exp(nu)
    
    y_pred = alpha + b1 * X[:, 0] + b2 * X[:, 1] + b3 * X[:, 2]
    resid = y - y_pred
    
    # method 1
    ll = t.logpdf(resid / sigma, df=nu) - np.log(sigma)
    
    # # method 2
    # n = len(y)
    # term1 = gammaln((nu+1)/2) - gammaln(nu/2)
    # term2 = -0.5 * np.log(np.pi * nu) - np.log(sigma)
    # term3 = - (nu+1)/2 * np.log(1 + (resid/sigma)**2 / nu)
    # ll = n * (term1 + term2) + np.sum(term3)
    
    return -np.sum(ll)

In [22]:
# method: algo to find min
# gtol: when to stop
result = minimize(neg_log_likelihood, initial_params, method='BFGS', options={'gtol': 1e-9})

In [23]:
params = result.x
alpha_est, b1_est, b2_est, b3_est = params[:4]
sigma_est = np.exp(params[4])
nu_est = np.exp(params[5])

In [24]:
print(f"Alpha: {alpha_est}")
print(f"B1: {b1_est}")
print(f"B2: {b2_est}")
print(f"B3: {b3_est}")
print(f"sigma: {sigma_est}")
print(f"nu: {nu_est}")

Alpha: 0.042634198339532914
B1: 0.9748885682120919
B2: 2.041192059685567
B3: 3.1548013572872162
sigma: 0.048548079528977696
nu: 4.59829274110107


Method 2

In [25]:
import statsmodels.miscmodels.tmodel as tmodel

In [26]:
model = tmodel.TLinearModel(y, X_bias)
result = model.fit()
print(result.summary())

running Tmodel initialize
Optimization terminated successfully.
         Current function value: -1.378814
         Iterations: 281
         Function evaluations: 451
                             TLinearModel Results                             
Dep. Variable:                      y   Log-Likelihood:                 137.88
Model:                   TLinearModel   AIC:                            -263.8
Method:            Maximum Likelihood   BIC:                            -248.1
Date:                Fri, 30 Jan 2026                                         
Time:                        11:24:46                                         
No. Observations:                 100                                         
Df Residuals:                      94                                         
Df Model:                           3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------